In [ ]:
!pip install tensorflow pandas scikit-learn numpy -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ai_pile_balanced.csv')

print(df.shape)
print(df['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = df['text'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_train),
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print("Tokenization Completed")

In [ ]:
print(MAX_WORDS)
print(MAX_LEN)

In [ ]:
print('X_train_pad' in globals())

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = df['text'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_train),
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print(X_train_pad.shape)
print(X_test_pad.shape)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

y_pred = (model.predict(X_test_pad) > 0.5).astype(int)

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=['Human', 'AI']
))

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Human', 'AI']
)

disp.plot()
plt.show()

In [ ]:
model.save('/content/drive/MyDrive/bilstm_ai_detector.keras')

import pickle

with open('/content/drive/MyDrive/bilstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("BiLSTM Model Saved!")

In [ ]:
from tensorflow.keras.models import load_model
import pickle

model = load_model('/content/drive/MyDrive/bilstm_ai_detector.keras')

with open('/content/drive/MyDrive/bilstm_tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print("BiLSTM Model Loaded!")

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 200

def predict_text(text):

    if len(text.split()) < 10:
        print("⚠️ Please enter at least 10 words.")
        return

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding='post',
        truncating='post'
    )

    prob = model.predict(padded, verbose=0)[0][0]

    if prob > 0.5:
        print(f"🤖 AI Generated ({prob*100:.2f}%)")
    else:
        print(f"👤 Human Written ({(1-prob)*100:.2f}%)")

In [ ]:
while True:

    text = input("\nEnter text (or quit): ")

    if text.lower() == "quit":
        break

    predict_text(text)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

y_pred = (model.predict(X_test_pad) > 0.5).astype(int)

print(classification_report(
    y_test,
    y_pred,
    target_names=['Human', 'AI']
))

In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test, verbose=1)

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Loss: {loss:.4f}")